# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library. All dataset elements are referenced by their Croissant `@id` fields to ensure consistent access.

### Dataset Source

The dataset source is provided via a Croissant schema URL.

- Croissant schema URL: [https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json)

> **Note:** All record sets and fields in this notebook are referenced exclusively by their `@id`.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # metadata is already a Croissant JsonLDObject (not a dict)

# Print dataset overview
print(f"Dataset Name: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"Dataset @id: {metadata['@id']}")
print(f"Keywords: {getattr(metadata, 'keywords', None)}")
print(f"Author(s) @id: {getattr(metadata, 'author', None)}")

## 2. Data Overview

Retrieve available record sets, their `@id`s, and fields. In Croissant, the list of available record sets is accessed via the schema, which is downloaded as part of the dataset object.

In [ ]:
# Explore available record sets (@id), fields (@id) and columns
from pprint import pprint

record_sets = dataset.record_sets
print(f"Found {len(record_sets)} record set(s):\n")
for rs in record_sets:
    print(f"Record Set Name: {getattr(rs, 'name', None)}")
    print(f"@id: {rs['@id']}")
    # List fields in this record set
    if hasattr(rs, 'fields') and rs.fields:
        print("  Fields:")
        for field in rs.fields:
            print(f"    - {getattr(field, 'name', None)} (@id: {field['@id']})")
            # If field is sourced from a column, print the column id
            if hasattr(field, 'column') and field.column is not None:
                print(f"      column @id: {field.column['@id']}")
    print()

## 3. Data Extraction

Load data from each record set into a Pandas DataFrame. All record sets and fields are referenced by `@id`.

In [ ]:
# Collect the record set @ids
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
print("Using record_set @ids:", record_set_ids)

# Extract records from each record set
dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        dataframes[record_set_id] = pd.DataFrame(records)
        print(f"DataFrame for {record_set_id}: {dataframes[record_set_id].shape[0]} rows, {dataframes[record_set_id].shape[1]} columns.")
        print("Columns:", dataframes[record_set_id].columns.tolist())
        display(dataframes[record_set_id].head(3))
    else:
        print(f"No records found for record set {record_set_id}.")

# For demonstration, pick the first record_set_id as main for EDA (if non-empty)
main_record_set_id = None
for rid in record_set_ids:
    if rid in dataframes:
        main_record_set_id = rid
        break
if main_record_set_id is not None:
    print(f"Proceeding with record set: {main_record_set_id}")
else:
    print("No record set with tabular data available.")

## 4. Exploratory Data Analysis (EDA)

We will analyze a numeric field from the chosen record set by `@id`. Filtering, normalization, and grouping will reference the column `@id` and operate only on existing DataFrame columns.

In [ ]:
# Identify numeric fields from the selected record set
if main_record_set_id is not None:
    main_rs = None
    for rs in dataset.record_sets:
        if rs['@id'] == main_record_set_id:
            main_rs = rs
            break
    numeric_field_id = None
    for field in getattr(main_rs, 'fields', []):
        if getattr(field, 'dataType', None) in ("Float", "Integer", "Number"):
            numeric_field_id = field['@id']
            break
    if numeric_field_id is None:
        # fallback: pick the first column with numeric dtype
        for c in dataframes[main_record_set_id].columns:
            if pd.api.types.is_numeric_dtype(dataframes[main_record_set_id][c]):
                numeric_field_id = c
                break
    if numeric_field_id:
        print(f"Using numeric field @id: {numeric_field_id}")
        # Set threshold as 10 (may adjust after inspecting data)
        threshold = 10
        df = dataframes[main_record_set_id]
        if numeric_field_id in df.columns:
            filtered_df = df[df[numeric_field_id] > threshold].copy()
            print(f"Filtered records with {numeric_field_id} > {threshold}:")
            display(filtered_df.head())
            # Normalization
            norm_col = f"{numeric_field_id}_normalized"
            filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
            print(f"Normalized {numeric_field_id} for filtered records:")
            display(filtered_df[[numeric_field_id, norm_col]].head())
            # Try grouping by another field (pick first categorical field in fields)
            group_field_id = None
            for field in getattr(main_rs, 'fields', []):
                if getattr(field, 'dataType', None) == "Text" and field['@id'] in df.columns:
                    group_field_id = field['@id']
                    break
            if group_field_id and group_field_id in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
                print(f"Grouped mean of {numeric_field_id} by {group_field_id}:")
                display(grouped_df.head())
            else:
                print("No suitable grouping field found.")
        else:
            print(f"Field {numeric_field_id} not present in DataFrame columns.")
    else:
        print("No numeric field found for EDA.")
else:
    print("No main record set selected for EDA.")

## 5. Visualization

Visualize the numeric field distribution and, where possible, relationships with categorical variables, all referenced by `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot histogram of the numeric field and group-by comparison
if main_record_set_id is not None and numeric_field_id and numeric_field_id in dataframes[main_record_set_id].columns:
    fig, axes = plt.subplots(1, 2, figsize=(12,4))
    # Histogram
    sns.histplot(dataframes[main_record_set_id][numeric_field_id], ax=axes[0], kde=True)
    axes[0].set_title(f"Distribution of {numeric_field_id}")
    axes[0].set_xlabel(numeric_field_id)
    # Boxplot by group if group_field_id is found
    if 'group_field_id' in locals() and group_field_id and group_field_id in dataframes[main_record_set_id].columns:
        sns.boxplot(data=dataframes[main_record_set_id], x=group_field_id, y=numeric_field_id, ax=axes[1])
        axes[1].set_title(f"{numeric_field_id} by {group_field_id}")
    plt.tight_layout()
    plt.show()
else:
    print("No numeric field available for visualization.")

## 6. Conclusion

- This notebook demonstrates loading and exploring a Croissant-described dataset using the `mlcroissant` Python library, with all references by `@id`.
- Step-by-step, we:
    - Loaded dataset metadata and structure
    - Inspected record sets, fields, and their `@id`s
    - Extracted tabular data into DataFrames
    - Performed initial EDA: filtering, normalization, and grouping
    - Visualized data distributions
- For custom analyses: refer to the `@id` fields provided in earlier sections for robust field referencing.

> For more information about the `mlcroissant` library or the FAIR^2 dataset, see the [Croissant documentation](https://mlcommons.org/croissant/) and [dataset description page](https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json).